# Join pieces into a square - with a very simple diffusion model

Same "pieces -> square" idea, but the join is now **rendered by a diffusion model** instead of pasting
pixels. We use **InstructPix2Pix** (~1B params, runs on a free Kaggle **T4**): give it an image + a
plain instruction ("arrange the bars into a square frame") and it edits the picture.

This is the smallest version of your real pipeline: **instruction-conditioned image editing**. It also
lets you *see the failure modes* you care about (drift, the rest of the scene not preserved, the edit
not landing) on a trivial input.

**Before running: Settings -> Accelerator -> GPU T4, Internet -> On.**

In [ ]:
# 1. Install. NOTE: Kaggle ships Pillow 11.3 which breaks diffusers (the `_Ink` ImportError),
#    so we pin Pillow < 11.3.
#    IMPORTANT: after this cell finishes, do  Run -> Restart session  then run all cells again,
#    so the pinned Pillow is the one actually loaded.
!pip install -q -U diffusers transformers accelerate gradio
!pip install -q "pillow<11.3"
import PIL; print("Pillow:", PIL.__version__, "(must be < 11.3 -- if not, Restart session)")

In [ ]:
# 2. Load the simple diffusion editor + make the starting "scattered bars" image
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler
from PIL import Image, ImageDraw
import gradio as gr

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix",
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
print("diffusion editor loaded")

def make_pieces_image(size=512):
    img = Image.new("RGB", (size, size), (240, 238, 232))
    d = ImageDraw.Draw(img)
    wood = (150, 100, 55)
    bars = [(40, 60, 240, 90),      # horizontal
            (300, 40, 330, 240),    # vertical
            (260, 420, 470, 450),   # horizontal
            (70, 280, 100, 470)]    # vertical
    for b in bars:
        d.rectangle(b, fill=wood, outline=(90, 60, 30), width=3)
    return img

start = make_pieces_image()
start

In [ ]:
# 3. Interface: edit the image with the diffusion model from an instruction

def assemble(image, instruction, steps, img_guidance, txt_guidance, seed):
    if image is None:
        return None
    image = image.convert("RGB").resize((512, 512))
    gen = torch.Generator("cuda").manual_seed(int(seed))
    out = pipe(
        instruction,
        image=image,
        num_inference_steps=int(steps),
        image_guidance_scale=float(img_guidance),
        guidance_scale=float(txt_guidance),
        generator=gen,
    ).images[0]
    return out

with gr.Blocks(title="Join into a square (diffusion)") as demo:
    gr.Markdown("## Pieces -> square, rendered by a diffusion model\nEdit the image with a plain instruction.")
    with gr.Row():
        with gr.Column():
            in_img = gr.Image(value=make_pieces_image(), type="pil", label="Start: scattered bars")
            instruction = gr.Textbox(value="arrange the four wooden bars into a square frame",
                                     label="Instruction")
            steps = gr.Slider(10, 50, value=20, step=1, label="Steps")
            img_guidance = gr.Slider(1.0, 2.5, value=1.5, step=0.1,
                                     label="Image guidance (higher = stay closer to input)")
            txt_guidance = gr.Slider(4.0, 12.0, value=7.5, step=0.5,
                                     label="Text guidance (higher = follow instruction harder)")
            seed = gr.Number(value=0, label="Seed", precision=0)
            run = gr.Button("Assemble with diffusion", variant="primary")
        with gr.Column():
            out_img = gr.Image(type="pil", label="Edited result")
    run.click(assemble, [in_img, instruction, steps, img_guidance, txt_guidance, seed], [out_img])

demo.launch(share=True)

## How to use
1. Run cell 1, then **Run -> Restart session**, then run all cells (GPU + Internet on).
2. Open the `gradio.live` link. Click **Assemble with diffusion**. Try editing the instruction and sliders.
3. Raise **image guidance** to keep the scene; raise **text guidance** to push the square harder.

## If you still hit the `_Ink` ImportError
It means the old Pillow is still loaded. Run cell 1, confirm it prints a version below 11.3, then
**Restart session** (Run menu) and run again. The restart is what matters - pip can downgrade the
files but a kernel that already imported the old Pillow keeps using it until restart.

## What to notice (ties back to your research)
- The edit will rarely produce a clean square. That is the point: a small instruction-editor is weakly
  controllable, and you will see your report's failure modes - the scene re-rendered, unrelated regions
  drift, the intended change only partly landing.
- This is the **rendering** layer. Compare it against the pixel-paste `connect()` from the main demo:
  paste is faithful but crude; diffusion is flexible but drifts. That trade-off is your core problem.
- To control it better: inpainting (mask only the join region) or reference-image conditioning (the
  target square as a reference) - the next experiments.